# 1D Prefix Sums, Integral Arrays, and Moving Averages

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Efficient numerical analysis frequently demands computing interval aggregations over streaming signals or multi-dimensional grids. By transforming a sequence into an **Integral Array** (Prefix Sum), range sum queries over arbitrary intervals $[a, b]$ can be computed in exact $O(1)$ time.

### Objectives
1. Formulate discrete integration and recurrence properties of prefix sums.
2. Implement constant-time $O(1)$ interval sum queries.
3. Compute Simple Moving Averages (SMA) and Exponential Moving Averages (EMA).
4. Apply moving average filtering to suppress telemetry noise in environmental data.


## 1. Discrete Integration: The Prefix Sum Recurrence

For a discrete 1D sequence $f(x)$ defined for indices $x \in \{0, 1, \dots, N-1\}$, the prefix sum array $F(x)$ represents the cumulative sum from index $0$ through $x$:

$$F(x) = \sum_{t=0}^x f(t)$$

With the linear recurrence relation:
$$F(x) = F(x - 1) + f(x), \quad \text{with } F(-1) = 0$$

### The $O(1)$ Range Sum Theorem
For any interval $[a, b]$ where $0 \le a \le b < N$:
$$\sum_{x=a}^b f(x) = F(b) - F(a - 1)$$


In [ ]:
def build_prefix_sum(arr: list[float]) -> list[float]:
    """Construct cumulative sum table in O(N) time and space."""
    if not arr:
        return []
    prefix = [0.0] * len(arr)
    prefix[0] = float(arr[0])
    for i in range(1, len(arr)):
        prefix[i] = prefix[i - 1] + float(arr[i])
    return prefix


def query_range_sum(prefix: list[float], a: int, b: int) -> float:
    """Query sum of arr[a...b] in O(1) time."""
    if a < 0 or b >= len(prefix) or a > b:
        raise IndexError(f"Invalid range [{a}, {b}] for prefix sum of length {len(prefix)}")
    if a == 0:
        return prefix[b]
    return prefix[b] - prefix[a - 1]


# Slide Example Verification:
# f(x) = [1, 8, 5, 7, 3, 5, 8, 3, 2, 9]
slide_f = [1, 8, 5, 7, 3, 5, 8, 3, 2, 9]
F = build_prefix_sum(slide_f)
print("f(x):", slide_f)
print("F(x):", [int(x) for x in F])

# Slide checks:
# F(3) = 1 + 8 + 5 + 7 = 21
# F(6) = 21 + 3 + 5 + 8 = 37
# Sum from x=4 to x=6: F(6) - F(3) = 37 - 21 = 16 (3 + 5 + 8 = 16)
assert int(F[3]) == 21
assert int(F[6]) == 37
range_4_6 = query_range_sum(F, 4, 6)
print("Sum from index 4 to 6:", int(range_4_6))
assert int(range_4_6) == 16

## 2. Moving Average Filtering via Prefix Sums

A Simple Moving Average (SMA) of window size $k$ replaces each reading with the arithmetic mean of its preceding $k$ observations. Using the prefix sum table, each window mean can be computed in constant time:

$$\text{SMA}_k(i) = \frac{F(i) - F(i - k)}{k}$$


In [ ]:
def simple_moving_average(arr: list[float], k: int) -> list[float]:
    """Compute k-period Simple Moving Average using O(1) prefix sum lookups."""
    if not arr or k <= 0 or k > len(arr):
        return []

    prefix = build_prefix_sum(arr)
    sma = []
    for i in range(k - 1, len(arr)):
        window_sum = query_range_sum(prefix, i - k + 1, i)
        sma.append(window_sum / k)
    return sma


def exponential_moving_average(arr: list[float], alpha: float) -> list[float]:
    """Compute Exponential Moving Average with smoothing factor alpha in (0, 1]."""
    if not arr:
        return []
    if not 0.0 < alpha <= 1.0:
        raise ValueError("Alpha must be in (0, 1].")

    ema = [arr[0]]
    for i in range(1, len(arr)):
        ema.append(alpha * arr[i] + (1.0 - alpha) * ema[-1])
    return ema


sample_signal = [10.0, 12.0, 14.0, 16.0, 18.0, 20.0]
sma_3 = simple_moving_average(sample_signal, k=3)
ema_smooth = exponential_moving_average(sample_signal, alpha=0.5)

print("Signal:           ", sample_signal)
print("SMA (k=3):        ", [round(x, 2) for x in sma_3])
print("EMA (alpha=0.5):  ", [round(x, 2) for x in ema_smooth])

# Window 0..2: (10 + 12 + 14)/3 = 12.0
# Window 1..3: (12 + 14 + 16)/3 = 14.0
assert sma_3 == [12.0, 14.0, 16.0, 18.0]

In [ ]:
import csv
from pathlib import Path

# Application to environmental telemetry
csv_file = Path("../data/sensor_stream.csv")
pressures = []
with csv_file.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pressures.append(float(row["pressure_kpa"]))

sma_pressures = simple_moving_average(pressures, k=5)
print(f"Original pressure observations: {len(pressures)}")
print(f"Smoothed SMA observations:      {len(sma_pressures)}")


# Compare signal variance before and after smoothing
def variance(vals: list[float]) -> float:
    mean = sum(vals) / len(vals)
    return sum((x - mean) ** 2 for x in vals) / len(vals)


raw_var = variance(pressures)
smoothed_var = variance(sma_pressures)
print(f"Raw Pressure Variance:      {raw_var:.6f}")
print(f"Smoothed Pressure Variance: {smoothed_var:.6f}")
assert smoothed_var < raw_var